# Fraud Detection in Financial Transactions Using Machine Learning

# Importing all required libraries


In [9]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix ,f1_score
import pickle

## Class: Data


In [10]:
class Data:

    def __init__(self, file_path):
        self.df = pd.read_csv(file_path)
        print(f"[INFO] Data loaded from: {file_path}")

    def display_head(self, n=5):
        return self.df.head(n)

    def display_tail(self, n=5):
        return self.df.tail(n)

    def display_sample(self, n=5):
        return self.df.sample(n)

    def get_info(self):
        self.df.info()

    def get_description(self):
        return self.df.describe()

    def get_shape(self):
        return self.df.shape

    def get_columns(self):
        return self.df.columns

    def get_mean(self):
        return self.df.mean(numeric_only=True)

    def get_median(self):
        return self.df.median(numeric_only=True)

    def get_mode(self):
        return self.df.mode()

    def get_data(self):
        return self.df

    def set_data(self, new_df: pd.DataFrame):
        self.df = new_df


## Class: DataPreprocessing


In [11]:
class DataPreprocessing:

    def __init__(self, dataframe):
        self.df = dataframe

    def select_columns(self, columns_to_keep: list):
        self.df = self.df[columns_to_keep]
        return self.df

    def check_null_values(self):
        return self.df.isnull().sum()

    def label_encode_column(self, column_name: str, new_column_name: str, value_map: dict):
        self.df[new_column_name] = self.df[column_name].map(value_map)
        return self.df


## Class: Model



In [12]:
class Model:

    def __init__(self, random_state: int = 42):
        self.random_state = random_state
        self.model = None
        self.scaler = None

    def split_data(self, X: pd.DataFrame, y: pd.Series, test_size: float = 0.2, stratify: bool = True):
        stratify_labels = y if stratify else None
        X_train, X_test, y_train, y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state=self.random_state,
            stratify=stratify_labels
        )
        return X_train, X_test, y_train, y_test

    def scale_data(self, X_train: pd.DataFrame, X_test: pd.DataFrame, scaler=None):
        if scaler is None:
            scaler = StandardScaler()
        self.scaler = scaler
        X_train_scaled = self.scaler.fit_transform(X_train)
        X_test_scaled = self.scaler.transform(X_test)
        return self.scaler, X_train_scaled, X_test_scaled

    def train_svm(self, X_train_scaled, y_train, kernel: str = 'linear', **kwargs):
        self.model = SVC(kernel=kernel, random_state=self.random_state, **kwargs)
        self.model.fit(X_train_scaled, y_train)
        return self.model

    def predict(self, X_test_scaled):
        y_pred = self.model.predict(X_test_scaled)
        return y_pred

    def evaluate_model(self, y_test, y_pred):
        print("\n--- Model Performance ---")
        print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
        print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
        print(f"Classification Report:\n{classification_report(y_test, y_pred)}")

    def get_model(self):
        return self.model

    def get_scaler(self):
        return self.scaler


## Class: PickleData










In [13]:
class PickleData:

    def save_object(self, obj, filename: str):
        with open(filename, 'wb') as f:
            pickle.dump(obj, f)
        print(f"Object saved to {filename}")

    def load_object(self, filename: str):
        with open(filename, 'rb') as f:
            obj = pickle.load(f)
        print(f"Object loaded from {filename}")
        return obj

    def predict_with_loaded_model(self, model_path: str, scaler_path: str, sample_data: pd.DataFrame):
        loaded_model = self.load_object(model_path)
        loaded_scaler = self.load_object(scaler_path)

        sample_data_scaled = loaded_scaler.transform(sample_data)
        prediction = loaded_model.predict(sample_data_scaled)
        return prediction


# Exploratory Data Summary Using Data Class

In [14]:
data = Data(file_path="data.csv")

print("\nHead:")
display(data.display_head())

print("\nTail:")
display(data.display_tail())

print("\nSample:")
display(data.display_sample())

print("\nInfo:")
data.get_info()

print("\nDescription:")
display(data.get_description())

print(f"\nShape: {data.get_shape()}")

print(f"\nColumns: {data.get_columns()}")

print(f"\nMean: \n{data.get_mean()}")

print(f"\nMedian: \n{data.get_median()}")


[INFO] Data loaded from: data.csv

Head:


,Unnamed: 0,TransactionID,AccountID,TransactionAmount,PreviousTransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,TransactionDate,is_fraud
0,0,TX000001,AC00128,14.09,11/04/2023 16:29,Debit,San Diego,D000380,162.198.218.92,M015,ATM,70,Doctor,81,1,5112.21,04/11/2024 8:08,0
1,1,TX000002,AC00455,376.24,27/06/2023 16:44,Debit,Houston,D000051,13.149.61.4,M052,ATM,68,Doctor,141,1,13758.91,04/11/2024 8:09,0
2,2,TX000003,AC00019,126.29,10/07/2023 18:16,Debit,Mesa,D000235,215.97.143.157,M009,Online,19,Student,56,2,1122.35,04/11/2024 8:07,1
3,3,TX000004,AC00070,184.50,05/05/2023 16:32,Debit,Raleigh,D000187,200.13.225.150,M002,Online,26,Student,25,5,8569.06,04/11/2024 8:09,1
4,4,TX000005,AC00411,13.45,16/10/2023 17:51,Credit,Atlanta,D000308,65.164.3.100,M091,Online,26,Student,198,1,7429.40,04/11/2024 8:06,0



Tail:


,Unnamed: 0,TransactionID,AccountID,TransactionAmount,PreviousTransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,TransactionDate,is_fraud
2507,2507,TX002508,AC00297,856.21,26/04/2023 17:09,Credit,Colorado Springs,D000625,21.157.41.17,M072,Branch,33,Doctor,109,1,12690.79,04/11/2024 8:11,0
2508,2508,TX002509,AC00322,251.54,22/03/2023 17:36,Debit,Tucson,D000410,49.174.157.140,M029,Branch,48,Doctor,177,1,254.75,04/11/2024 8:11,0
2509,2509,TX002510,AC00095,28.63,21/08/2023 17:08,Debit,San Diego,D000095,58.1.27.124,M087,Branch,56,Retired,146,1,3382.91,04/11/2024 8:08,0
2510,2510,TX002511,AC00118,185.97,24/02/2023 16:24,Debit,Denver,D000634,21.190.11.223,M041,Online,23,Student,19,3,1776.91,04/11/2024 8:12,1
2511,2511,TX002512,AC00009,243.08,14/02/2023 16:21,Credit,Jacksonville,D000215,59.127.135.25,M041,Online,24,Student,93,2,131.25,04/11/2024 8:07,1



Sample:


,Unnamed: 0,TransactionID,AccountID,TransactionAmount,PreviousTransactionDate,TransactionType,Location,DeviceID,IP Address,MerchantID,Channel,CustomerAge,CustomerOccupation,TransactionDuration,LoginAttempts,AccountBalance,TransactionDate,is_fraud
2026,2026,TX002027,AC00069,8.81,22/05/2023 16:57,Debit,Dallas,D000462,35.67.21.23,M046,Online,57,Doctor,165,1,13523.65,04/11/2024 8:09,0
73,73,TX000074,AC00110,233.92,20/10/2023 16:54,Credit,Jacksonville,D000295,152.140.239.181,M046,Online,26,Student,280,4,6537.62,04/11/2024 8:08,1
380,380,TX000381,AC00329,112.05,26/10/2023 16:16,Credit,Phoenix,D000156,116.44.12.250,M078,ATM,26,Student,159,4,1122.88,04/11/2024 8:08,1
1684,1684,TX001685,AC00219,291.27,19/06/2023 17:58,Debit,Detroit,D000119,184.59.28.72,M097,Branch,77,Retired,121,1,6878.86,04/11/2024 8:10,0
1690,1690,TX001691,AC00442,12.18,20/04/2023 18:50,Debit,New York,D000326,190.152.148.249,M088,Branch,76,Retired,77,1,4909.24,04/11/2024 8:07,0



Info:
<class 'pandas.DataFrame'>
RangeIndex: 2512 entries, 0 to 2511
Data columns (total 18 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Unnamed: 0               2512 non-null   int64  
 1   TransactionID            2512 non-null   str    
 2   AccountID                2512 non-null   str    
 3   TransactionAmount        2512 non-null   float64
 4   PreviousTransactionDate  2512 non-null   str    
 5   TransactionType          2512 non-null   str    
 6   Location                 2512 non-null   str    
 7   DeviceID                 2512 non-null   str    
 8   IP Address               2512 non-null   str    
 9   MerchantID               2512 non-null   str    
 10  Channel                  2512 non-null   str    
 11  CustomerAge              2512 non-null   int64  
 12  CustomerOccupation       2512 non-null   str    
 13  TransactionDuration      2512 non-null   int64  
 14  LoginAttempts            251

,Unnamed: 0,TransactionAmount,CustomerAge,TransactionDuration,LoginAttempts,AccountBalance,is_fraud
count,2512.000000,2512.000000,2512.000000,2512.000000,2512.000000,2512.000000,2512.000000
mean,1255.500000,297.593778,44.673965,119.643312,1.661624,5114.302966,0.252787
std,725.296261,291.946243,17.792198,69.963757,1.107949,3900.942499,0.434696
min,0.000000,0.260000,18.000000,10.000000,1.000000,101.250000,0.000000
25%,627.750000,81.885000,27.000000,63.000000,1.000000,1504.370000,0.000000
50%,1255.500000,211.140000,45.000000,112.500000,1.000000,4735.510000,0.000000
75%,1883.250000,414.527500,59.000000,161.000000,2.000000,7678.820000,1.000000
max,2511.000000,1919.110000,80.000000,300.000000,5.000000,14977.990000,1.000000



Shape: (2512, 18)

Columns: Index(['Unnamed: 0', 'TransactionID', 'AccountID', 'TransactionAmount',
       'PreviousTransactionDate', 'TransactionType', 'Location', 'DeviceID',
       'IP Address', 'MerchantID', 'Channel', 'CustomerAge',
       'CustomerOccupation', 'TransactionDuration', 'LoginAttempts',
       'AccountBalance', 'TransactionDate', 'is_fraud'],
      dtype='str')

Mean: 
Unnamed: 0             1255.500000
TransactionAmount       297.593778
CustomerAge              44.673965
TransactionDuration     119.643312
LoginAttempts             1.661624
AccountBalance         5114.302966
is_fraud                  0.252787
dtype: float64

Median: 
Unnamed: 0             1255.50
TransactionAmount       211.14
CustomerAge              45.00
TransactionDuration     112.50
LoginAttempts             1.00
AccountBalance         4735.51
is_fraud                  0.00
dtype: float64


# Data preprocessing


In [15]:
# Data preprocessing
preprocessor = DataPreprocessing(data.get_data())

selected_columns = ['TransactionAmount', 'CustomerAge', 'LoginAttempts', 'AccountBalance', 'Channel', 'is_fraud']
df_selected = preprocessor.select_columns(selected_columns)
data.set_data(df_selected) 

print("\nNull values after selection:")
display(preprocessor.check_null_values())

preprocessor.label_encode_column('Channel', 'ChannelEncoded', {'ATM': 0, 'Online': 1, 'Branch': 2})

df_encoded = preprocessor.label_encode_column('Channel', 'ChannelEncoded', {'ATM': 0, 'Online': 1, 'Branch': 2})
data.set_data(df_encoded)

print("\nHead after preprocessing:")
display(data.display_head())



Null values after selection:


TransactionAmount    0
CustomerAge          0
LoginAttempts        0
AccountBalance       0
Channel              0
is_fraud             0
dtype: int64


Head after preprocessing:


,TransactionAmount,CustomerAge,LoginAttempts,AccountBalance,Channel,is_fraud,ChannelEncoded
0,14.09,70,1,5112.21,ATM,0,0
1,376.24,68,1,13758.91,ATM,0,0
2,126.29,19,2,1122.35,Online,1,1
3,184.50,26,5,8569.06,Online,1,1
4,13.45,26,1,7429.40,Online,0,1


# Model Training and Evaluation



In [16]:
# Model Training and Evaluation
feature_cols = ['TransactionAmount', 'CustomerAge', 'AccountBalance', 'ChannelEncoded', 'LoginAttempts']
target_col = 'is_fraud'

# Use the preprocessor instance from Cell 10 or get data from data
current_df_for_ml = data.get_data()
X, y = current_df_for_ml[feature_cols], current_df_for_ml[target_col]

ml_model_handler = Model()

X_train, X_test, y_train, y_test = ml_model_handler.split_data(X, y)
scaler_obj, X_train_scaled, X_test_scaled = ml_model_handler.scale_data(X_train, X_test)

# RF config that reached ~0.69 class-1 F1 in quick search
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_leaf=2,
    class_weight='balanced_subsample',
    random_state=42
 )
rf_model.fit(X_train_scaled, y_train)

# Threshold tuning fixed from best quick run
y_proba = rf_model.predict_proba(X_test_scaled)[:, 1]
y_pred = (y_proba >= 0.62).astype(int)

# Keep object references compatible with existing pickling cell
ml_model_handler.model = rf_model
ml_model_handler.scaler = scaler_obj

print("\n--- Model Performance ---")
print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")
print(f"Classification Report:\n{classification_report(y_test, y_pred)}")


--- Model Performance ---
Accuracy: 0.8668
Confusion Matrix:
[[358  18]
 [ 49  78]]
Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.95      0.91       376
           1       0.81      0.61      0.70       127

    accuracy                           0.87       503
   macro avg       0.85      0.78      0.81       503
weighted avg       0.86      0.87      0.86       503



#  Pickling and Prediction


In [17]:
#  Pickling and Prediction
pickler = PickleData()

pickler.save_object(ml_model_handler.get_model(), 'svm_model.pkl')
pickler.save_object(ml_model_handler.get_scaler(), 'scaler.pkl')

# Sample 1
sample_1_data = {'TransactionAmount': [1500], 'CustomerAge': [35],
                 'AccountBalance': [25000], 'ChannelEncoded': [0], 'LoginAttempts': [1]}
random_sample_1 = pd.DataFrame(sample_1_data)
prediction_1 = pickler.predict_with_loaded_model(
    model_path='svm_model.pkl',
    scaler_path='scaler.pkl',
    sample_data=random_sample_1[feature_cols] # Ensure column order
)
print(f"Prediction for random sample 1: {prediction_1[0]}")

# Sample 2
sample_2_data = {'TransactionAmount': [500], 'CustomerAge': [25],
                 'AccountBalance': [25000], 'ChannelEncoded': [0], 'LoginAttempts': [3]}
random_sample_2 = pd.DataFrame(sample_2_data)
prediction_2 = pickler.predict_with_loaded_model(
    model_path='svm_model.pkl',
    scaler_path='scaler.pkl',
    sample_data=random_sample_2[feature_cols] # Ensure column order
)
print(f"Prediction for random sample 2: {prediction_2[0]}")

Object saved to svm_model.pkl
Object saved to scaler.pkl
Object loaded from svm_model.pkl
Object loaded from scaler.pkl
Prediction for random sample 1: 0
Object loaded from svm_model.pkl
Object loaded from scaler.pkl
Prediction for random sample 2: 0
